# V4.1 — Frozen V4 + Author-Aware Retrieval

Train a small author residual branch on top of frozen V4, then evaluate it on the exact point-in-time A2 protocol. The combined vectors remain ANN-compatible. Epoch checkpoints are written to Drive so an interrupted runtime can resume.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile
DRIVE_ROOT=Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT=DRIVE_ROOT/'data/gold/adaptive/v1'
BASE_GOLD_ROOT=DRIVE_ROOT/'data/gold/two_tower/v1_colab'
PROTOCOL_DIR=ADAPTIVE_ROOT/'evaluation_protocol/v2'
OUTPUT_DIR=DRIVE_ROOT/'artifacts/adaptive/v4_1_author'
WORK_ROOT=Path('/content/recsys_v4_1_author')
SOURCE_ROOT=WORK_ROOT/'runtime'; V4_DIR=WORK_ROOT/'v4'; REPORT_DIR=WORK_ROOT/'report'
for path in [SOURCE_ROOT,V4_DIR,REPORT_DIR,OUTPUT_DIR]: path.mkdir(parents=True,exist_ok=True)
print('checkpoint dir:',OUTPUT_DIR)

In [ ]:
required={'pyarrow':'pyarrow','pandas':'pandas','torch':'torch','huggingface_hub':'huggingface_hub>=0.32'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import pandas as pd, torch
assert torch.cuda.is_available(),'Select an L4 or A100 GPU runtime.'
print('GPU:',torch.cuda.get_device_name(0))
required_paths=[ADAPTIVE_ROOT/'sequence_examples/train',ADAPTIVE_ROOT/'sequence_examples/validation',BASE_GOLD_ROOT/'item_features/static',PROTOCOL_DIR/'queries.parquet']
missing_paths=[str(path) for path in required_paths if not path.exists()]
assert not missing_paths,'Missing inputs:\n'+'\n'.join(missing_paths)

In [ ]:
from huggingface_hub import hf_hub_download
HF_REPO_ID='khoa1505/recsys-artifacts'
RUNTIME_EXPECTED_SHA256='8e9f002cac5e2c802686ecc82ee8dcd0817fd266311e9d97ebeb4a80fd7ba56f'
V4_EXPECTED_SHA256='4ad49b3e6f9f8b5d45a6c92f31d8d9b27f558e1746613efb44a23db3cd456a26'
try: HF_TOKEN=userdata.get('HF_TOKEN')
except Exception: HF_TOKEN=getpass('Paste Hugging Face READ token: ')
assert HF_TOKEN and HF_TOKEN.startswith('hf_')
runtime_zip=Path(hf_hub_download(HF_REPO_ID,'runtime/adaptive_eval_runtime.zip',repo_type='model',token=HF_TOKEN))
v4_cached=Path(hf_hub_download(HF_REPO_ID,'v4/v4_inference_fp16.pt',repo_type='model',token=HF_TOKEN))
def sha256(path):
    digest=hashlib.sha256()
    with path.open('rb') as stream:
        while block:=stream.read(8*1024*1024): digest.update(block)
    return digest.hexdigest()
assert sha256(runtime_zip)==RUNTIME_EXPECTED_SHA256
assert sha256(v4_cached)==V4_EXPECTED_SHA256
if SOURCE_ROOT.exists(): shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None; archive.extractall(SOURCE_ROOT)
v4_target=V4_DIR/'best_model.pt'
if v4_target.exists() or v4_target.is_symlink(): v4_target.unlink()
v4_target.symlink_to(v4_cached.resolve())
TRAIN_SCRIPT=SOURCE_ROOT/'scripts/train_adaptive_author_branch.py'
EVAL_SCRIPT=SOURCE_ROOT/'scripts/evaluate_adaptive_author_branch.py'
assert TRAIN_SCRIPT.exists() and EVAL_SCRIPT.exists()
print('artifact integrity: PASS')

## Train or resume the author branch

In [ ]:
train_command=[sys.executable,str(TRAIN_SCRIPT),'--adaptive-root',str(ADAPTIVE_ROOT),'--base-gold-root',str(BASE_GOLD_ROOT),'--v4-dir',str(V4_DIR),'--output-dir',str(OUTPUT_DIR),'--epochs','4','--batch-size','1024','--learning-rate','0.0008','--embedding-dim','32','--max-train-queries','0','--max-validation-queries','100000','--device','cuda','--resume']
train_log=Path('/content/v4_1_author_training.log')
process=subprocess.Popen(train_command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with train_log.open('w') as log:
    for line in process.stdout: print(line,end=''); log.write(line); log.flush()
code=process.wait()
if code!=0: raise RuntimeError('Training failed:\n'+'\n'.join(train_log.read_text(errors='replace').splitlines()[-160:]))
assert (OUTPUT_DIR/'best_model.pt').exists()
print('training complete')

## Evaluate V4.1 on the locked A2 protocol

In [ ]:
eval_command=[sys.executable,str(EVAL_SCRIPT),'--adaptive-root',str(ADAPTIVE_ROOT),'--base-gold-root',str(BASE_GOLD_ROOT),'--v4-dir',str(V4_DIR),'--author-dir',str(OUTPUT_DIR),'--protocol-dir',str(PROTOCOL_DIR),'--output-dir',str(REPORT_DIR),'--device','cuda']
eval_log=Path('/content/v4_1_author_evaluation.log')
process=subprocess.Popen(eval_command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with eval_log.open('w') as log:
    for line in process.stdout: print(line,end=''); log.write(line); log.flush()
code=process.wait()
if code!=0: raise RuntimeError('Evaluation failed:\n'+'\n'.join(eval_log.read_text(errors='replace').splitlines()[-160:]))

In [ ]:
summary=pd.read_csv(REPORT_DIR/'summary.csv')
display(summary[['variant','hitrate@10','ndcg@10','hitrate@100','hitrate@1000','ndcg@1000']])
report=json.loads((REPORT_DIR/'v4_1_author_evaluation.json').read_text())
print('author scale:',report['author_scale'])

In [ ]:
archive_path=Path('/content/v4_1_author_report.zip')
if archive_path.exists(): archive_path.unlink()
with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in REPORT_DIR.rglob('*'):
        if path.is_file(): archive.write(path,Path('v4_1_author_report')/path.relative_to(REPORT_DIR))
    for path in [OUTPUT_DIR/'config.json',OUTPUT_DIR/'metrics.json',OUTPUT_DIR/'training_history.csv',train_log,eval_log]:
        if path.exists(): archive.write(path,Path('run_metadata')/path.name)
persistent=OUTPUT_DIR/'v4_1_author_report.zip'; shutil.copy2(archive_path,persistent)
print('checkpoint:',OUTPUT_DIR/'best_model.pt')
print('report:',persistent,archive_path.stat().st_size,'bytes')
files.download(str(archive_path))